<a href="https://colab.research.google.com/github/OssamaSijbesma/uu-data-wrangling-assignment-1/blob/main/wc_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Assignment 1: Data Extraction & Integration](https://hqahtan.github.io/Teaching/infomdw/assignments/Assignment_1_26_27.html)

In [48]:
import sqlite3
import pandas as pd
import requests
from collections import defaultdict

# Set constants
GOOGLE_COLAB = True  # When using Google Colab set to True.
DATABASE_URL = "https://github.com/OssamaSijbesma/uu-data-wrangling-assignment-1/raw/refs/heads/main/wc_database.db"

## Attach database

In [49]:
conn = None

if GOOGLE_COLAB:
    # Connect to in memory database
    conn = sqlite3.connect(":memory:")

    # Fetch database from online data source
    response = requests.get(DATABASE_URL)

    # Load serialized database
    conn.deserialize(response.content)
else:
    # Connect to local database file
    conn = sqlite3.connect("./wc_database.db")

# Create cursor
cursor = conn.cursor()

## Execute queries

In [50]:
def query_database(
    sql: str, connection: sqlite3.Connection
) -> None | list[sqlite3.Row]:
    # Create cursor
    cursor = conn.cursor()
    try:
        # Execute query
        cursor.execute(sql)

        # Fetch the output of the query
        result = cursor.fetchall()
    except sqlite3.Error as error:
        # Print error if query failed
        print(error)
        return None
    finally:
        # Close cursor to prevent memory leaks
        cursor.close()

    # Return the output as a list
    return result

In [51]:
sql_query = """SELECT pe.first_name,
    pe.last_name,
    t.country_name AS team,
    s.name AS stadium,
    g.minute
FROM goal g
    JOIN player p ON g.player_id = p.player_id
    JOIN person pe ON p.person_id = pe.person_id
    JOIN team t ON p.team_id = t.team_id
    JOIN match m ON g.match_id = m.match_id
    JOIN stadium s ON m.stadium_id = s.stadium_id
ORDER BY g.match_id,
    g.minute;"""

query_database(sql_query, conn)

[('Memphis', 'Depay', 'Netherlands', 'Johan Cruyff Arena', 23),
 ('Harry', 'Kane', 'England', 'Johan Cruyff Arena', 67),
 ('Lautaro', 'Martinez', 'Argentina', 'Camp Nou', 35),
 ('Bernardo', 'Silva', 'Spain', 'Camp Nou', 72),
 ('Harry', 'Kane', 'England', 'Wembley Stadium', 41),
 ('Lautaro', 'Martinez', 'Argentina', 'Wembley Stadium', 83)]

In [52]:
sql_query = """SELECT t.country_name,
    COUNT(g.goal_id) AS total_goals
FROM goal g
    JOIN player p ON g.player_id = p.player_id
    JOIN team t ON p.team_id = t.team_id
GROUP BY t.team_id,
    t.country_name
ORDER BY total_goals DESC;
"""

query_database(sql_query, conn)

[('England', 2), ('Argentina', 2), ('Netherlands', 1), ('Spain', 1)]

In [53]:
sql_query = "Trigger an error"

query_database(sql_query, conn)

near "Trigger": syntax error


## Read table into Pandas data frame

In [54]:
df = pd.read_sql_query("SELECT last_name, first_name, birth_date FROM PERSON", conn)
df.head()

,last_name,first_name,birth_date
0,Koeman,Ronald,1963-03-21
1,Van Dijk,Virgil,1991-07-08
2,De Jong,Frenkie,1997-05-12
3,Depay,Memphis,1994-02-13
4,Gakpo,Cody,1999-05-07


## Hash column of strings

In [55]:
def hash_column(column: list[str], table_cardinality=7):
    # Create hash table
    hash_table = defaultdict(list)

    # Loop through column
    for x in column:
        # Hash to positive number
        hashed = abs(hash(x))
        # Set the contents in the bin based on the hash and modulo
        hash_table[hashed % table_cardinality].append(x)

    # Return the table
    return hash_table

In [56]:
# Execute hash function on data frame column
table = hash_column(df["last_name"])

# Loop to the hashed table
for key in table:
    # Print key and content
    print(f"{key}: {table[key]}")

2: ['Koeman', 'Depay', 'Silva', 'Muller', 'Brown']
0: ['Van Dijk', 'Wilson']
6: ['De Jong', 'Martinez', 'Martinez']
5: ['Gakpo', 'Rice', 'Kane', 'Bellingham', 'Garcia']
1: ['Santos']
3: ['Taylor']
4: ['Oliver']


## Collision rate

In [57]:
N = len(df["last_name"])
B = len(table)

collision_rate = (N - B) / N
print(f"Total elements: {N}")
print(f"Occupied buckets: {B}")
print(f"Collision rate: {collision_rate:.2%}")

Total elements: 18
Occupied buckets: 7
Collision rate: 61.11%


## Close database

In [58]:
# Close connection
conn.close()